In [1]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.5.1+cu121
12.1
True
NVIDIA TITAN Xp


In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU Count: 1
GPU Name: NVIDIA TITAN Xp


In [3]:
import bitsandbytes as bnb

print(bnb.__version__)

0.49.2


In [4]:
from huggingface_hub import login
login()
# print("No Error")

/mnt/sda1/osint_intern/miniconda3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '69354be9d7724dd447774e5b', 'name': 'RaunakRajR', 'fullname': 'Raunak Raj', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/69354be9d7724dd447774e5b/sERmIcXEC2fBPiLcW5q5y.png', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-07-24T06:46:15.000Z'}}


In [6]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-1b-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)



Loading weights: 100%|██████████████████████████████████████████| 340/340 [00:00<00:00, 9402.66it/s]
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for use! \n\nI’m designed to take text and images as input and produce text as output. \n\nIt’s nice to meet you – how can I help you today?'}]}]

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loaded")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████████████████████████| 340/340 [00:00<00:00, 1281.40it/s]


Loaded


In [10]:
print(next(model.parameters()).device)

cpu


In [11]:
import torch

print(torch.cuda.memory_allocated()/1024**3, "GB")
print(torch.cuda.memory_reserved()/1024**3, "GB")

1.9195890426635742 GB
1.9453125 GB


In [12]:
import pandas as pd

In [13]:
# df=pd.read_csv("/mnt/sda1/osint_intern/SA_Group/Raunak/data.csv")
#Reassesment 
df=pd.read_csv("/mnt/sda1/osint_intern/SA_Group/Raunak/Culture_Aware_Sentiment_Analysis/Analysis/Gemma_4B_reassesment.csv")
df
df.columns


Index(['Sentence', 'Region', 'State', 'Sentiment'], dtype='str')

In [ ]:
df.head(10)


In [14]:
print("Done")
df.describe()

Done


,Sentence,Region,State,Sentiment
count,114,114,114,114
unique,22,6,33,1
top,कुछ लोग विवाह से पहले लंबे समय तक डेटिंग करना ...,Northern Cultural Region,Chhattisgarh,PARSE_ERROR
freq,13,22,11,114


In [15]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [16]:
print(next(model.parameters()).device)

cpu


In [17]:
import torch

print(torch.cuda.memory_allocated()/1024**3, "GB")
print(torch.cuda.memory_reserved()/1024**3, "GB")

1.9195890426635742 GB
1.9375 GB


In [18]:
!nvidia-smi

Fri Jun 26 16:34:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA TITAN Xp                Off |   00000000:01:00.0 Off |                  N/A |
| 54%   82C    P2            172W /  250W |   11929MiB /  12288MiB |     28%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
messages = [
    {
        "role": "user",
        "content": prompt
    }
]

chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

prompts.append(chat_prompt)
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_new_tokens=80
)

generated = outputs[0][inputs["input_ids"].shape[1]:]

print(
    tokenizer.decode(
        generated,
        skip_special_tokens=True
    )
)

NameError: name 'prompt' is not defined

In [22]:
import os
import re
import json
import torch
import pandas as pd
from tqdm.auto import tqdm

# =====================================================
# PROMPT
# =====================================================

PROMPT_TEMPLATE = """
You are an expert sentiment classifier.

Your task is to classify sentiment by considering:
- cultural context
- regional practices
- social norms
- geographical sensitivity
- local interpretation

A sentence may express different sentiment in different locations.

STRICT RULES:
- Output ONLY valid JSON
- No explanation
- No markdown
- No extra text

SCHEMA:
{{
    "sentiment":"positive | neutral | negative"
}}

Location:
{location}

Text:
{text}

Return ONLY JSON.
"""

# =====================================================
# SETTINGS
# =====================================================

BATCH_SIZE = 17
PROGRESS_FILE = "reassesd_inference_progress.csv"
FINAL_FILE = "all_reassesd_predictions.csv"

# =====================================================
# LOAD PREVIOUS PROGRESS IF AVAILABLE
# =====================================================

if os.path.exists(PROGRESS_FILE):

    print(
        f"Found existing progress file: "
        f"{PROGRESS_FILE}"
    )

    df = pd.read_csv(
        PROGRESS_FILE
    )

    print(
        "Previous progress loaded."
    )

else:

    print(
        "No previous progress found."
    )

    if "Sentiment" not in df.columns:
        df["Sentiment"] = None


Found existing progress file: reassesd_inference_progress.csv
Previous progress loaded.


In [23]:

# =====================================================
# JSON PARSER
# =====================================================

def extract_sentiment(response):

    response = response.strip().lower()

    try:
        match = re.search(r"\{.*?\}", response, re.DOTALL)

        if match:
            obj = json.loads(match.group())

            sentiment = obj.get("sentiment", "").lower().strip()

            if sentiment in [
                "positive",
                "neutral",
                "negative"
            ]:
                return sentiment

    except:
        pass

    if re.search(r"\bpositive\b", response):
        return "positive"

    if re.search(r"\bneutral\b", response):
        return "neutral"

    if re.search(r"\bnegative\b", response):
        return "negative"

    return "PARSE_ERROR"


In [24]:
# =====================================================
# GEMMA 3 INSTRUCT INFERENCE
# =====================================================

def classify_batch(batch_df):

    sentiments = []

    for _, row in batch_df.iterrows():

        location = (
            f"{row['State']} - "
            f"{row['Region']}"
        )

        prompt = PROMPT_TEMPLATE.format(
            location=location,
            text=str(row["Sentence"])
        )

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        decoded = tokenizer.decode(
            generated,
            skip_special_tokens=True
        )

        sentiment = extract_sentiment(decoded)

        if sentiment == "PARSE_ERROR":

            print("\n==============================")
            print("PARSE ERROR")
            print("------------------------------")
            print(decoded)
            print("==============================\n")

        sentiments.append(sentiment)

    return sentiments

In [25]:
# =====================================================
# DETERMINE STARTING POINT
# =====================================================

if "Sentiment" not in df.columns:
    df["Sentiment"] = None

completed_mask = (
    df["Sentiment"].notna()
    & (df["Sentiment"] != "PARSE_ERROR")
)

start_idx = int(
    completed_mask.sum()
)

print(
    f"Completed rows: "
    f"{start_idx}/{len(df)}"
)

# =====================================================
# MAIN PROCESSING LOOP
# =====================================================

rows_to_process = df[
    df["Sentiment"].isna()
    | (df["Sentiment"] == "PARSE_ERROR")
]

for start in tqdm(
    range(0, len(rows_to_process), BATCH_SIZE),
    desc="Reassessing"
):

    batch_idx = rows_to_process.index[start:start+BATCH_SIZE]

    batch_df = df.loc[batch_idx]

    preds = classify_batch(batch_df)

    df.loc[batch_idx, "Sentiment"] = preds

    df.to_csv(
        PROGRESS_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved progress: "
        f"{end}/{len(df)}"
    )

# =====================================================
# FINAL SAVE
# =====================================================

df.to_csv(
    FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nSaved final file -> "
    f"{FINAL_FILE}"
)

# =====================================================
# SAVE REGION-WISE FILES
# =====================================================

for region in df["Region"].dropna().unique():

    region_df = df[
        df["Region"] == region
    ].copy()

    filename = (
        region.replace(
            " ",
            "_"
        )
        + "_Reassesed_Gemma_4B.csv"
    )

    region_df.to_csv(
        filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved -> {filename}"
    )

print(
    "\nFinished Successfully."
)



Completed rows: 16/114


Reassessing:   0%|                                                            | 0/6 [43:06<?, ?it/s]


NameError: name 'end' is not defined

In [ ]:
import gc
import torch

# Delete large objects
del model
del tokenizer

# If they exist
# del pipe
# del inputs
# del outputs

# Run Python garbage collection
gc.collect()

# Free unused CUDA memory
torch.cuda.empty_cache()

# Reset CUDA memory statistics
torch.cuda.reset_peak_memory_stats()

print("Memory released.")